# yfinance API Inspection

This notebook is for testing how Yahoo Finance / `yfinance` data looks before adding anything to the main eMFer app.

Goal:
- Search Yahoo Finance by name
- Inspect returned symbols, names, asset types, exchanges, and currencies
- Fetch historical price data for selected symbols
- Convert yfinance output into the same basic shape eMFer already uses: `date`, `nav`, `fund_name`


## 1. Install yfinance

Run this once if `yfinance` is not already installed in your notebook environment.

In [ ]:
# Uncomment and run this cell if yfinance is not installed.
# %pip install yfinance

## 2. Import libraries

In [1]:
import pandas as pd
import yfinance as yf

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 80)

/Users/niloy/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


## 3. Search Yahoo Finance by name

Try queries like:
- `Vanguard S&P 500`
- `Apple`
- `Vanguard Total Stock Market`
- `Nippon India ETF Nifty`
- `ICICI Prudential Nifty ETF`

In [2]:
search_query = "bandhan small cap"

search = yf.Search(search_query, max_results=10)
search_results = search.quotes

search_results

[{'exchange': 'BSE',
  'shortname': '0P0001J6FW.BO',
  'quoteType': 'MUTUALFUND',
  'symbol': '0P0001J6FW.BO',
  'index': 'quotes',
  'score': 20001.0,
  'typeDisp': 'Fund',
  'longname': 'Bandhan Small Cap Dir IDCW-T',
  'exchDisp': 'Bombay',
  'isYahooFinance': True},
 {'exchange': 'BSE',
  'shortname': '0P0001J6FZ.BO',
  'quoteType': 'MUTUALFUND',
  'symbol': '0P0001J6FZ.BO',
  'index': 'quotes',
  'score': 20001.0,
  'typeDisp': 'Fund',
  'longname': 'Bandhan Small Cap Reg IDCW-T',
  'exchDisp': 'Bombay',
  'isYahooFinance': True},
 {'exchange': 'BSE',
  'shortname': '0P0001J6FU.BO',
  'quoteType': 'MUTUALFUND',
  'symbol': '0P0001J6FU.BO',
  'index': 'quotes',
  'score': 20001.0,
  'typeDisp': 'Fund',
  'longname': 'Bandhan Small Cap Dir Gr',
  'exchDisp': 'Bombay',
  'isYahooFinance': True},
 {'exchange': 'BSE',
  'shortname': '0P0001J6FX.BO',
  'quoteType': 'MUTUALFUND',
  'symbol': '0P0001J6FX.BO',
  'index': 'quotes',
  'score': 20001.0,
  'typeDisp': 'Fund',
  'longname': 'Ba

## 4. Convert search results into a clean table

Yahoo search returns a list of dictionaries. This table keeps only the fields that are useful for eMFer selection.

In [3]:
def clean_yfinance_search_results(search_results):
    rows = []

    for item in search_results:
        rows.append({
            "symbol": item.get("symbol"),
            "name": item.get("longname") or item.get("shortname"),
            "quote_type": item.get("quoteType"),
            "exchange": item.get("exchange"),
            "currency": item.get("currency"),
            "market": item.get("market"),
        })

    return pd.DataFrame(rows)


search_table = clean_yfinance_search_results(search_results)
search_table

,symbol,name,quote_type,exchange,currency,market
0,0P0001J6FW.BO,Bandhan Small Cap Dir IDCW-T,MUTUALFUND,BSE,None,None
1,0P0001J6FZ.BO,Bandhan Small Cap Reg IDCW-T,MUTUALFUND,BSE,None,None
2,0P0001J6FU.BO,Bandhan Small Cap Dir Gr,MUTUALFUND,BSE,None,None
3,0P0001J6FX.BO,Bandhan Small Cap Reg Gr,MUTUALFUND,BSE,None,None
4,0P0001J6FV.BO,Bandhan Small Cap Dir IDCW-P,MUTUALFUND,BSE,None,None
5,0P0001J6FY.BO,Bandhan Small Cap Reg IDCW-P,MUTUALFUND,BSE,None,None


## 5. Fetch historical data for one selected symbol

Even when the user searches by name, yfinance fetches history using the selected symbol.

Try examples:
- `VOO` - US ETF
- `VFIAX` - US mutual fund
- `AAPL` - US stock
- `NIFTYBEES.NS` - Indian ETF

In [4]:
selected_symbol = "0P0001J6FU.BO	"

ticker = yf.Ticker(selected_symbol)
history = ticker.history(period="5y")

history.head()

,Open,High,Low,Close,Volume,Dividends,Stock Splits,Capital Gains
Date,,,,,,,,
2021-06-30 00:00:00+05:30,20.520000,20.520000,20.520000,20.520000,0,0.0,0.0,0.0
2021-07-01 00:00:00+05:30,20.709999,20.709999,20.709999,20.709999,0,0.0,0.0,0.0
2021-07-02 00:00:00+05:30,20.799999,20.799999,20.799999,20.799999,0,0.0,0.0,0.0
2021-07-05 00:00:00+05:30,20.990000,20.990000,20.990000,20.990000,0,0.0,0.0,0.0
2021-07-06 00:00:00+05:30,20.840000,20.840000,20.840000,20.840000,0,0.0,0.0,0.0


## 6. Inspect metadata

This helps us understand what eMFer can show to users during selection.

In [5]:
info = ticker.info

metadata = {
    "symbol": selected_symbol,
    "long_name": info.get("longName"),
    "short_name": info.get("shortName"),
    "quote_type": info.get("quoteType"),
    "currency": info.get("currency"),
    "exchange": info.get("exchange"),
    "country": info.get("country"),
}

metadata

{'symbol': '0P0001J6FU.BO\t',
 'long_name': 'Bandhan Small Cap Dir Gr',
 'short_name': '0P0001J6FU.BO',
 'quote_type': 'MUTUALFUND',
 'currency': 'INR',
 'exchange': 'BSE',
 'country': None}

## 7. Convert yfinance history into eMFer shape

Current eMFer calculations expect:

```text
date | nav | fund_name
```

For yfinance, we can use `Close` as the first simple version. Later we can decide whether to prefer adjusted prices depending on what yfinance returns for each asset.

In [6]:
def convert_yfinance_history_to_emfer_shape(history, metadata):
    fund_name = metadata.get("long_name") or metadata.get("short_name") or metadata.get("symbol")

    emfer_history = history.reset_index().copy()
    emfer_history = emfer_history.rename(columns={"Date": "date", "Close": "nav"})

    emfer_history["date"] = pd.to_datetime(emfer_history["date"]).dt.tz_localize(None)
    emfer_history["nav"] = pd.to_numeric(emfer_history["nav"])
    emfer_history["fund_name"] = fund_name
    emfer_history["symbol"] = metadata.get("symbol")
    emfer_history["quote_type"] = metadata.get("quote_type")
    emfer_history["currency"] = metadata.get("currency")
    emfer_history["source"] = "yfinance"

    return emfer_history[["date", "nav", "fund_name", "symbol", "quote_type", "currency", "source"]]


emfer_yfinance_history = convert_yfinance_history_to_emfer_shape(history, metadata)
emfer_yfinance_history.head()

,date,nav,fund_name,symbol,quote_type,currency,source
0,2021-06-30,20.520000,Bandhan Small Cap Dir Gr,0P0001J6FU.BO\t,MUTUALFUND,INR,yfinance
1,2021-07-01,20.709999,Bandhan Small Cap Dir Gr,0P0001J6FU.BO\t,MUTUALFUND,INR,yfinance
2,2021-07-02,20.799999,Bandhan Small Cap Dir Gr,0P0001J6FU.BO\t,MUTUALFUND,INR,yfinance
3,2021-07-05,20.990000,Bandhan Small Cap Dir Gr,0P0001J6FU.BO\t,MUTUALFUND,INR,yfinance
4,2021-07-06,20.840000,Bandhan Small Cap Dir Gr,0P0001J6FU.BO\t,MUTUALFUND,INR,yfinance


## 8. Quick data quality check

This is similar to what eMFer already does before rolling return calculation.

In [7]:
quality_summary = {
    "rows": len(emfer_yfinance_history),
    "start_date": emfer_yfinance_history["date"].min(),
    "end_date": emfer_yfinance_history["date"].max(),
    "missing_nav_rows": emfer_yfinance_history["nav"].isna().sum(),
    "zero_or_negative_nav_rows": (emfer_yfinance_history["nav"] <= 0).sum(),
}

quality_summary

{'rows': 1226,
 'start_date': Timestamp('2021-06-30 00:00:00'),
 'end_date': Timestamp('2026-06-29 00:00:00'),
 'missing_nav_rows': np.int64(0),
 'zero_or_negative_nav_rows': np.int64(0)}

## 9. Try multiple searches quickly

Use this section to compare how search results differ for funds, ETFs, and stocks.

In [8]:
queries = [
    "Vanguard S&P 500",
    "Apple",
    "Vanguard Total Stock Market",
    "Nippon India ETF Nifty",
]

all_search_rows = []

for query in queries:
    results = yf.Search(query, max_results=5).quotes
    table = clean_yfinance_search_results(results)
    table.insert(0, "search_query", query)
    all_search_rows.append(table)

all_search_results = pd.concat(all_search_rows, ignore_index=True)
all_search_results

,search_query,symbol,name,quote_type,exchange,currency,market
0,Vanguard S&P 500,VOO,Vanguard S&P 500 ETF,ETF,PCX,None,None
1,Vanguard S&P 500,VFV.TO,Vanguard S&P 500 Index ETF,ETF,TOR,None,None
2,Vanguard S&P 500,VOOG,Vanguard S&P 500 Growth Index Fund ETF Shares,ETF,PCX,None,None
3,Vanguard S&P 500,VUSA.L,Vanguard S&P 500 UCITS ETF,ETF,LSE,None,None
4,Vanguard S&P 500,VUSA.AS,Vanguard S&P 500 UCITS ETF,ETF,AMS,None,None
5,Apple,AAPL,Apple Inc.,EQUITY,NMS,None,None
6,Apple,APLE,"Apple Hospitality REIT, Inc.",EQUITY,NYQ,None,None
7,Apple,APC.DE,Apple Inc.,EQUITY,GER,None,None
8,Apple,APC.F,Apple Inc.,EQUITY,FRA,None,None
9,Apple,2788.T,"Apple International Co., Ltd.",EQUITY,JPX,None,None
